# Stage C — 25M-class A100 capacity and horizon matrix
Measures the exact 25.27M-parameter model before any long training. It runs horizons 2/3/4 on identical A100 hardware and appends every completed probe to Drive.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='8579b06b641e442191277a61ecfd3644cf1593c4'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
# Fresh A100-only evidence folder; the earlier c8 folder preserves the T4 failure.
RUN_NAME='c8_a100_capacity_25m_model_a100'
BLOCK_COUNT=11
D_MODEL=512
NUM_HEADS=8
PERSISTENT_TOKENS=4
MEMORY_DEPTH=1

In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess,sys
mountpoint=Path('/content/drive')
try:
    if not (mountpoint/'MyDrive').is_dir(): drive.mount(str(mountpoint),force_remount=True,timeout_ms=120000)
except ValueError as error:
    raise RuntimeError('Google Drive did not mount. Restart the Colab runtime, reconnect Drive, accept the authorization prompt, then rerun this cell.') from error
if not (mountpoint/'MyDrive').is_dir(): raise RuntimeError('Google Drive is not ready at /content/drive/MyDrive; do not continue.')
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
device_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA device'
if 'A100' not in device_name.upper(): raise RuntimeError(f'Notebook 02 requires an A100; Colab assigned {device_name}. Runtime > Change runtime type, select A100, then restart and rerun.')
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
output=f'{DRIVE_ROOT}/runs/{RUN_NAME}'
Path(output).mkdir(parents=True,exist_ok=True)
PROTOCOL=repo/'studies'/'stage_c_ecoli_escherichia_medium_25m_v1'/'protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study'/'stage_c_ecoli_escherichia_medium_25m_v1'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
def logged(label,command):
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',label,'--repo',str(repo),'--',*command],check=True)
logged('hardware_preflight',['seqtrainer-titans-stage-c-hardware-preflight','--require','A100','--output',f'{output}/hardware.json'])

In [ ]:
# Keep each full-geometry probe below Colab's long-child-process watchdog and preserve prior results.
# reference_fp32 is deliberately excluded: its unaccelerated functional recurrence is a correctness oracle, not a capacity workload.
for horizon in ('2','3','4'):
    for variant in ('exact_sdpa_fp32','exact_sdpa_bfloat16'):
        logged(f'a100_{variant}_h{horizon}',['seqtrainer-titans-stage-c-capacity','--dataset-dir',DATASET_DIR,'--output-dir',output,'--require','A100','--horizons',horizon,'--variants',variant,'--steps','2','--batch-size','1','--block-count',str(BLOCK_COUNT),'--d-model',str(D_MODEL),'--num-heads',str(NUM_HEADS),'--persistent-tokens',str(PERSISTENT_TOKENS),'--memory-depth',str(MEMORY_DEPTH),'--append'])
subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id','reference_feasibility','--evidence-tier','engineering','--artifact',output],check=True)
print('SHARE THIS DIRECTORY:',output)